# CFNA convergence run — drive the base model to its floor

Single purpose: take the existing `cfna_base_large` checkpoint (last best
**1.377 bpb**) to the lowest held-out bits/byte this 11M model + corpus can
reach, using the **learning-rate ladder**: train at the current LR until the
best stops improving for ~15 min (a plateau *at that LR*), then drop the LR
3x and continue. Convergence = a 10x LR drop buys < ~0.02 bpb.

Everything is resume-safe and auto-backed-up every 30 min, so a disconnect
costs nothing — just reopen and re-run cell 4. The LR fix (commit 991a6e1)
means `--lr` on resume actually takes effect, which is what makes the ladder
work; do NOT use an older notebook copy for this.

In [ ]:
# 1) Setup — clone latest, verify GPU + AMP safety
import os
REPO = "LMMinier/nueronce"; BRANCH = "claude/cfna-neural-core-verify-ldvgn3"
if not os.path.exists("nueronce"):
    !git clone --branch $BRANCH https://github.com/$REPO nueronce
%cd nueronce
!git pull
%pip install -q numpy 'datasets>=2.19,<4' pytest cryptography cffi
import torch; print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
!python -m pytest tests/test_gpu_amp.py -q

In [ ]:
# 2) Restore the base checkpoint + history from Drive
from google.colab import drive
from pathlib import Path
import shutil, json
drive.mount("/content/drive")
SRC = Path("/content/drive/MyDrive/CFNA_checkpoints"); Path("checkpoints").mkdir(exist_ok=True)
for n in ["cfna_base_large.pt", "cfna_base_large.pt.json", "cfna_base_large_best.pt"]:
    if (SRC/n).exists(): shutil.copy2(SRC/n, Path("checkpoints")/n); print("restored:", n)
hist = json.load(open("checkpoints/cfna_base_large.pt.json"))["history"]
best = min(h["heldout_bpb"] for h in hist)
print(f"\nresuming from step {hist[-1]['step']} | best held-out bpb so far: {best:.4f}")

In [ ]:
# 3) Corpus — rebuild the training corpus this checkpoint was trained on.
#    (Colab disks are ephemeral; the manifest/text must exist to resume.)
#    ~10-15 min; HF cache makes reruns fast. Larger corpus = lower floor,
#    so this uses a bigger budget than the original session.
!python scripts/dump_corpus_stack.py --out corpus_large --phase 2 \
    --target-bytes 900000000 --val-every 20
!du -sh corpus_large

In [ ]:
# 4) THE LADDER — set LR for this rung, launch background training + auto-backup.
#    Rung schedule (advance when cell 4a shows the best flat for ~15 min):
#        rung 1: LR = 3e-4   (or 5e-4 to match the original run)
#        rung 2: LR = 1e-4
#        rung 3: LR = 3e-5
#        rung 4: LR = 1e-5   (stop when this buys < ~0.02 bpb)
#    Just change LR below and re-run this cell for each rung; --resume + the
#    lr-on-resume fix carry the weights forward and apply the new LR.
import os, subprocess, threading, time, shutil
from pathlib import Path
LR = "1e-4"          # <-- edit per rung
MINUTES = "170"
BATCH = "32"          # A100/L4; drop to 16 on T4
cmd = ["python", "-u", "scripts/train_checkpoint.py", "--corpus", "corpus_large",
       "--minutes", MINUTES, "--seq", "192", "--batch", BATCH, "--lr", LR,
       "--amp", "--resume", "--out", "checkpoints/cfna_base_large.pt"]
Path("logs").mkdir(exist_ok=True); LOG = Path("logs/conv.log")
PROC = subprocess.Popen(cmd, stdout=LOG.open("w"), stderr=subprocess.STDOUT,
                        text=True, start_new_session=True)
def _bk():
    dst = Path("/content/drive/MyDrive/CFNA_checkpoints")
    while PROC.poll() is None:
        time.sleep(1800)
        for n in ["cfna_base_large.pt", "cfna_base_large.pt.json", "cfna_base_large_best.pt"]:
            if Path(f"checkpoints/{n}").exists(): shutil.copy2(f"checkpoints/{n}", dst/n)
        print("auto-backup", time.strftime("%H:%M"), flush=True)
threading.Thread(target=_bk, daemon=True).start()
print(f"convergence training launched at LR={LR}, PID {PROC.pid}")

In [ ]:
# 4a) Monitor + plateau detector (rerun anytime)
import re
lines = LOG.read_text(errors="replace").splitlines()
print("alive:", PROC.poll() is None, "" if PROC.poll() is None else f"(exit {PROC.returncode})")
print("\n".join(lines[-8:]))
b = [float(m.group(1)) for l in lines if (m := re.search(r'held-out bpb ([0-9.]+)', l))]
if b:
    best = min(b); since = len(b) - 1 - (len(b) - 1 - b[::-1].index(best))
    print(f"\nthis rung: {b[0]:.3f} -> {b[-1]:.3f} | best {best:.3f} | "
          f"{len(b)-1-since} evals ({(len(b)-1-since)*50} steps) since best")
    if len(b) - 1 - since >= 18:
        print(">>> PLATEAU at this LR — drop LR 3x in cell 4 and re-run for the next rung.")

In [ ]:
# 4b) OPTIONAL live text probe (sampled) — watch quality sharpen as bpb falls
import torch
from cfna.model import CFNAModel, ModelConfig
ck = torch.load('checkpoints/cfna_base_large_best.pt', map_location='cpu', weights_only=False)
m = CFNAModel(ModelConfig(**ck['config'])); m.load_state_dict(ck['state_dict']); m.eval()
print(f"probing best (bpb {ck.get('best_heldout_bpb','?')})\n")
for p in ["The nature of human understanding is", "Once upon a time"]:
    out = m.generate(p.encode(), max_new=90, temperature=0.8, greedy=False)
    print(f">>> {p}\n{(p.encode()+out[len(p.encode()):] if isinstance(out,(bytes,bytearray)) else out).decode('utf-8','replace')}\n")

In [ ]:
# 5) Bank to Drive (auto-backup already runs; this forces a final sync)
import shutil
from pathlib import Path
dst = Path("/content/drive/MyDrive/CFNA_checkpoints")
for n in ["cfna_base_large.pt", "cfna_base_large.pt.json", "cfna_base_large_best.pt", "logs/conv.log"]:
    if Path(f"checkpoints/{n}" if not n.startswith("logs") else n).exists():
        src = Path(f"checkpoints/{n}" if not n.startswith("logs") else n)
        shutil.copy2(src, dst/src.name); print("backed up:", src.name)

## When is it converged?

Stop the ladder when a **10x LR drop buys less than ~0.02 bpb** — that is the
floor for this 11M model on this corpus (estimate: 1.25-1.35 given you were at
1.377 mid-run). Two ways to go lower after that, in order of value:

1. **Bigger corpus** — raise `--target-bytes` in cell 3 to 2-4 GB; a
   data-starved model's floor drops with more data before more compute.
2. **Bigger model** — the 35M rung (`scripts/train_checkpoint.py --preset
   base_35m`) has a lower floor than the 11M can ever reach. Convergence of
   the small model is the signal to climb the ladder, not to keep grinding.

Whichever you do, this checkpoint is the warm-start / comparison point, safely
in Drive.